# LastDance full Search-First SFT
Enable Internet and GPU. Attach `amnpham/zalochallenge` and a private input containing the precomputed `train_gipformer.json`. Add Kaggle Secrets `GEMINI_API_KEY` or `GEMINI_API_KEY_1` through `GEMINI_API_KEY_6`. The notebook runs preparation, fresh LoRA SFT, then one Val200 benchmark. Checkpoints and reports are saved under `/kaggle/working/lastdance_full`.
Gemini receives the request, history, schemas and simulator observations of eligible training cases. Enable the fallback only with inputs you are authorized to send to Gemini.

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys

WORK = Path("/kaggle/working/lastdance_full")
REPO = Path("/kaggle/working/LastDance")
DATA_ROOT = Path("/kaggle/input/datasets/amnpham/zalochallenge/train+public_test")
TRANSCRIPTS = ""  # Set an explicit path if multiple train_gipformer.json files are attached.
USE_GEMINI = True
GEMINI_MODEL = "gemini-3.5-flash-lite"
SECRET_NAMES = ["GEMINI_API_KEY"] + [f"GEMINI_API_KEY_{i}" for i in range(1, 7)]
BATCH = 8
SAVE_STEPS = 20
RUN_TAG = "full_fresh_lora"

if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/AmnO-O/LastDance.git", str(REPO)], check=True)
subprocess.run(["git", "pull", "--ff-only"], cwd=REPO, check=True)
if not (DATA_ROOT / "train" / "train.jsonl").is_file():
    raise FileNotFoundError(DATA_ROOT / "train" / "train.jsonl")
paths = [Path(TRANSCRIPTS)] if TRANSCRIPTS else sorted(Path("/kaggle/input").rglob("train_gipformer.json"))
if len(paths) != 1 or not paths[0].is_file():
    raise ValueError("Attach train_gipformer.json and set TRANSCRIPTS to its path.")
env = dict(os.environ, LIFEMATE_DATA_ROOT=str(DATA_ROOT), LIFEMATE_TRANSCRIPTS=str(paths[0]),
           LIFEMATE_WORK=str(WORK), PYTHONUNBUFFERED="1")

def stage(*args, extra_env=None):
    subprocess.run([sys.executable, "-m", "pipeline.run_search_sft", *args],
                   cwd=REPO, env={**env, **(extra_env or {})}, check=True)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-c", "import torch; assert torch.cuda.is_available(); print(torch.cuda.device_count(), 'GPU(s)')"], check=True)


In [ ]:
def prepare():
    args = ["prepare", "--full-train", "--transcripts", "mixed"]
    if not USE_GEMINI:
        stage(*args)
        return
    from kaggle_secrets import UserSecretsClient
    client = UserSecretsClient()
    keys = []
    for name in SECRET_NAMES:
        try:
            value = client.get_secret(name)
        except Exception:
            continue
        if value and value.strip():
            keys.append(value.strip())
    if not keys:
        raise RuntimeError("Add at least one Gemini API key in Kaggle Secrets.")
    stage(*args, "--gemini-fallback", "--gemini-model", GEMINI_MODEL,
          extra_env={"LIFEMATE_GEMINI_KEYS": json.dumps(keys)})

prepare()


In [ ]:
import math

manifest = json.loads((WORK / "gold_replay_manifest.json").read_text())
assert manifest["full_train"] and manifest["transcripts"] == "mixed"
rows = manifest["examples"]["train"]["count"]
if not rows:
    raise RuntimeError("No accepted training rows.")
steps = math.ceil(rows / (BATCH * SAVE_STEPS)) * SAVE_STEPS
print(f"{rows} accepted rows, {steps} optimizer steps, batch {BATCH}")
stage("train", "--steps", str(steps), "--save-steps", str(SAVE_STEPS),
      "--batch", str(BATCH), "--learning-rate", "1e-5", "--run-tag", RUN_TAG)


In [ ]:
stage("val", "--adapter", str(WORK / RUN_TAG / "adapter"))
print((WORK / "val_report.json").read_text(encoding="utf-8"))
